# Artifact Lineage Slice

Journey purpose: prove the narrow Phase 8 artifact-lineage slice over
candidate-centered links.

Notebook use: `proof`

This deep-dive notebook shows the first recovered artifact-backed provenance
path: source artifact -> derived dataset -> analysis result -> candidate
lineage report.

Related docs, code, tests, and evidence:

- `docs/plans/0001_successor_roadmap.md`
- `docs/plans/0002_phase8_artifact_lineage_shape.md`
- `docs/adr/0008-start-artifact-lineage-with-a-narrow-three-kind-model-and-candidate-centered-links.md`
- `src/onto_canon6/artifacts/`
- `src/onto_canon6/surfaces/lineage_report.py`
- `tests/artifacts/test_artifact_service.py`


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

SEARCH_ROOTS = [
    Path.cwd().resolve(),
    Path('~/projects/onto-canon6').expanduser(),
]

PROJECT_ROOT = None
for start in SEARCH_ROOTS:
    candidate = start
    while True:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            PROJECT_ROOT = candidate
            break
        if candidate.parent == candidate:
            break
        candidate = candidate.parent
    if PROJECT_ROOT is not None:
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate onto-canon6 repo root from notebook cwd or ~/projects/onto-canon6"
    )

for candidate_path in (
    PROJECT_ROOT / "src",
    PROJECT_ROOT.parent / "llm_client",
    Path('~/projects/llm_client').expanduser(),
):
    if candidate_path.exists():
        candidate_str = str(candidate_path)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)

from pprint import pprint
from tempfile import TemporaryDirectory

from onto_canon6.artifacts import ArtifactLineageService
from onto_canon6.pipeline import ReviewService
from onto_canon6.surfaces import LineageReportService


## Phase 1: Candidate Setup

Purpose: persist one reviewed candidate that will become the subject of
artifact-backed provenance.

Input -> output: `raw_source_context -> candidate_assertion`

Acceptance criteria:
- One candidate assertion is persisted through the existing review store.
- The candidate remains the primary review-governed subject.
- The phase runs live.

Status: `proven`
Execution mode: `live`


In [2]:
workspace = TemporaryDirectory()
workspace_path = Path(workspace.name)
review_service = ReviewService(
    db_path=workspace_path / 'review.sqlite3',
    overlay_root=workspace_path / 'ontology_overlays',
    default_acceptance_policy='record_only',
)
submission = review_service.submit_candidate_assertion(
    payload={
        'predicate': 'oc:uses_reviewed_artifact_support',
        'roles': {
            'subject': [{'entity_id': 'ent:account:x'}],
            'object': [{'entity_id': 'ent:campaign:network'}],
        },
    },
    profile_id='default',
    profile_version='1.0.0',
    submitted_by='analyst:notebook',
    source_kind='report_file',
    source_ref='reports/campaign_summary.txt',
    source_label='campaign summary report',
    source_text='Account X appears central in the campaign network.',
)
assert submission.candidate.candidate_id.startswith('cand_')
pprint(submission.candidate.model_dump())


{'candidate_id': 'cand_c04844bd93c640c79b3201bd',
 'claim_text': None,
 'evidence_spans': (),
 'normalized_payload': {'predicate': 'oc:uses_reviewed_artifact_support',
                        'roles': {'object': [{'entity_id': 'ent:campaign:network',
                                              'kind': 'entity'}],
                                  'subject': [{'entity_id': 'ent:account:x',
                                               'kind': 'entity'}]}},
 'payload': {'predicate': 'oc:uses_reviewed_artifact_support',
             'roles': {'object': [{'entity_id': 'ent:campaign:network',
                                   'kind': 'entity'}],
                       'subject': [{'entity_id': 'ent:account:x',
                                    'kind': 'entity'}]}},
 'payload_hash': 'sha256:ba177eac813cb215418d358bb0d72155a4f06ba7c698ddac123203b6204244d0',
 'profile': {'profile_id': 'default', 'profile_version': '1.0.0'},
 'proposal_ids': (),
 'provenance': {'source_artifact': {'conten

## Phase 2: Artifact Registration And Lineage

Purpose: register explicit source, derived, and analysis artifacts and connect
them through lineage edges.

Input -> output: `candidate_assertion -> artifact_registry_state`

Acceptance criteria:
- Three explicit artifact kinds are persisted.
- Artifact-to-artifact lineage is explicit.
- Candidate-centered support links are explicit and live.

Status: `proven`
Execution mode: `live`


In [3]:
artifact_service = ArtifactLineageService(db_path=review_service.store.db_path)
source_artifact = artifact_service.register_artifact(
    artifact_kind='source',
    uri='reports/campaign_summary.txt',
    label='campaign summary report',
    metadata={'format': 'txt'},
)
dataset_artifact = artifact_service.register_artifact(
    artifact_kind='derived_dataset',
    uri='derived/retweet_graph.graphml',
    label='retweet graph',
    metadata={'format': 'graphml'},
)
analysis_artifact = artifact_service.register_artifact(
    artifact_kind='analysis_result',
    uri='analysis/centrality_scores.json',
    label='centrality scores',
    metadata={'metric': 'betweenness'},
)
artifact_service.add_lineage_edge(
    parent_artifact_id=source_artifact.artifact_id,
    child_artifact_id=dataset_artifact.artifact_id,
)
artifact_service.add_lineage_edge(
    parent_artifact_id=dataset_artifact.artifact_id,
    child_artifact_id=analysis_artifact.artifact_id,
)
artifact_service.link_candidate_artifact(
    candidate_id=submission.candidate.candidate_id,
    artifact_id=source_artifact.artifact_id,
    support_kind='quoted_from',
    reference_detail='source paragraph 1',
)
artifact_service.link_candidate_artifact(
    candidate_id=submission.candidate.candidate_id,
    artifact_id=analysis_artifact.artifact_id,
    support_kind='supported_by_analysis',
    reference_detail='betweenness 0.89',
)


CandidateArtifactLinkRecord(candidate_id='cand_c04844bd93c640c79b3201bd', artifact_id='art_7aea7120c4444a959b157862', support_kind='supported_by_analysis', reference_detail='betweenness 0.89', created_at='2026-03-18T17:37:55.960290+00:00')

## Phase 3: Candidate Lineage Report

Purpose: expose the candidate, direct support links, and ancestor artifact
chain through one inspectable report surface.

Input -> output: `artifact_registry_state -> candidate_lineage_report`

Acceptance criteria:
- The report shows direct candidate links.
- The report includes source, derived dataset, and analysis artifacts.
- The report includes the lineage chain without hidden metadata blobs.

Status: `proven`
Execution mode: `live`


In [4]:
lineage_report = LineageReportService(artifact_service=artifact_service).build_candidate_report(
    candidate_id=submission.candidate.candidate_id,
)
assert [link.support_kind for link in lineage_report.direct_artifact_links] == [
    'quoted_from',
    'supported_by_analysis',
]
assert {artifact.artifact_kind for artifact in lineage_report.artifacts} == {
    'source',
    'derived_dataset',
    'analysis_result',
}
assert {(edge.parent_artifact_id, edge.child_artifact_id) for edge in lineage_report.lineage_edges} == {
    (source_artifact.artifact_id, dataset_artifact.artifact_id),
    (dataset_artifact.artifact_id, analysis_artifact.artifact_id),
}
pprint(lineage_report.model_dump())


{'artifacts': ({'artifact_id': 'art_1ae79e3f66754a5ca7e4d3fd',
                'artifact_kind': 'source',
                'created_at': '2026-03-18T17:37:55.939720+00:00',
                'fingerprint': None,
                'label': 'campaign summary report',
                'metadata': {'format': 'txt'},
                'uri': 'reports/campaign_summary.txt'},
               {'artifact_id': 'art_2fcb5f6efa034430bec921ba',
                'artifact_kind': 'derived_dataset',
                'created_at': '2026-03-18T17:37:55.942757+00:00',
                'fingerprint': None,
                'label': 'retweet graph',
                'metadata': {'format': 'graphml'},
                'uri': 'derived/retweet_graph.graphml'},
               {'artifact_id': 'art_7aea7120c4444a959b157862',
                'artifact_kind': 'analysis_result',
                'created_at': '2026-03-18T17:37:55.945911+00:00',
                'fingerprint': None,
                'label': 'centrality scores',
    

In [5]:
workspace.cleanup()